Now that we have ground truth data, we can evaluate how well our search
retrieves the correct documents.

For each question in our ground truth dataset, we run search. Then we
check whether the results include the correct document.

In [3]:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
print(df_ground_truth.shape)
df_ground_truth.tail()

(720, 2)


,question,document
715,Where can I find the latest dlt workshop mater...,e00ceb24be
716,Is there a directory for the current Open-Sour...,e00ceb24be
717,I’m looking for the 2026 dlt workshop instruct...,e00ceb24be
718,What folder should I open to get the current d...,e00ceb24be
719,Has the workshop page moved to a new location ...,e00ceb24be


For search evaluation, we only need the search part of the RAG pipeline. We don't need to call the LLM yet.

In [4]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [5]:
import sys
sys.path.append("..")

In [7]:
from src.ingest import load_faq_data, build_index

Load the documents and build a minsearch index:

In [8]:
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

Wrap the search call in a function called `text_search`. The name is
deliberate. Later we'll write `vector_search` or a hybrid version and
run the exact same evaluation on it. Everything downstream only needs a
function that takes a query and returns results, so we can swap one for
another. That mirrors how RAG works: the retrieval step doesn't care
which search function sits behind it.


In [9]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

### Collecting relevance data

Start with one ground truth record:

In [10]:
q = ground_truth[0]
q

{'question': 'I just found this course — is it still okay to join now, or am I too late?',
 'document': '74eb249bbf'}

Run search for this question:

In [11]:
doc_id = q["document"]
results = text_search(query=q["question"])

First, compare the retrieved document IDs with the correct document ID:

In [12]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
c2903069a0 == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False
5cc511f85b == 74eb249bbf: False


Then turn this comparison into a relevance list. In this lesson, relevance means whether a retrieved document is the correct document for this question.

In [13]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

This gives a list of 0 and 1 values. 1 means the retrieved document has the same ID as the correct document.

Put this logic into a function:

In [14]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

For the first ground truth record, the relevance list is:

In [15]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)

I just found this course — is it still okay to join now, or am I too late?


[1, 0, 0, 0, 0]

The correct document was the first search result.

Here are two more examples from the generated ground truth data.

For this question:

In [16]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)

Where is the live stream link for workshop sessions usually announced before they start?


[1, 0, 0, 0, 0]

In [17]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

How do I keep track of the LLM Zoomcamp syllabus, homework deadlines, and my progress in one place?


[1, 0, 0, 0, 0]

The correct document was found at the first position again.

Now do the same thing for all ground truth questions:


In [19]:
from tqdm.auto import tqdm

In [20]:
def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

Call it for the first 15 ground truth questions:

In [21]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [22]:
relevance_total_text

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0]]

Each entry in `relevance_total_text` is a relevance list. This is
enough to check that the function works before we run it for the full
dataset.

Next, make the relevance functions generic. We start with text search,
but later we may want to evaluate vector search, hybrid search, or
another retrieval method. The relevance logic is the same. Only the
search function changes.

In [23]:
def compute_relevance(q, search_function):
    
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

The total relevance function gets a `search_function` too.

We need to provide it explicitly:

In [24]:
def compute_relevance_total(ground_truth, search_function):
    
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

Use it with `text_search` on the same sample:

In [25]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0]]

Now run it for all ground truth questions:

In [26]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/720 [00:00<?, ?it/s]

### Search Evaluation Metrics

In [27]:
sample = relevance_total[:15]
sample

[[1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0]]

Hit Rate (also called Recall@k) measures the fraction of queries where the correct document appears anywhere in the results:

In [28]:
cnt = 0

for line in sample:
    if 1 in line:
        cnt = cnt + 1

cnt / len(sample)

0.8666666666666667

Put the same logic into a function:

In [29]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [31]:
hit_rate(sample)

0.8666666666666667

### Mean Reciprocal Rank (MRR)

Hit Rate tells us if we found the right document, but not where it was.

MRR also considers the position.

For each query, the score is based on the rank of the first correct
document:

- position 1: score is 1.0
- position 2: score is 0.5
- position 3: score is 0.333
- not found: score is 0

Let's calculate MRR:

In [35]:
total_score = 0.0

for line in sample:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score / len(sample)

0.711111111111111

MRR is the average of these scores across all queries. It rewards
systems that put the correct document near the top.

Hit Rate is the upper bound for MRR. In practice, MRR is usually
smaller because some correct documents are found below the first
position.

In [36]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [37]:
mrr(sample)

0.711111111111111

Wrap the metrics in a reusable evaluation function:

In [39]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

We can evaluate any search function:

In [40]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/720 [00:00<?, ?it/s]

{'hit_rate': 0.8416666666666667, 'mrr': 0.711342592592592}

Search metrics tell us whether retrieval works. Next, we'll use these
metrics to tune the search parameters.

### Search Parameter Tuning

Instead of guessing which settings are better, we measure them on the
ground truth dataset.

So far we've boosted `question` to 3.0. The idea was that a query should
match the FAQ question. That kind of match should count for more than
matching the answer text. It sounds reasonable. But it's a guess, and now
we can check it against data instead of trusting it.

This is the main benefit of offline evaluation. We change one parameter,
run the same questions again, and see whether the metric moves. The
dataset stays fixed, so the comparison is fair.

#### Trying different boosts

Start with a search function where the question boost is configurable:

In [41]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Evaluate several boost values:

In [42]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/720 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8888888888888888, 'mrr': 0.7910648148148146}


  0%|          | 0/720 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8847222222222222, 'mrr': 0.7752083333333328}


  0%|          | 0/720 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8416666666666667, 'mrr': 0.711342592592592}


  0%|          | 0/720 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.7944444444444444, 'mrr': 0.6730787037037032}


  0%|          | 0/720 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7680555555555556, 'mrr': 0.6471759259259255}


Increasing the question boost makes the metrics worse, not better. The
best value here is `1.0`, no boost at all. That's already the opposite of
what the intuition predicted.

But this is only one parameter. We can also tune `answer` and `section`
together with `question`.

Define a search function with all three boosts:

In [43]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Now do a small grid search:

In [44]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/720 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/720 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/720 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/720 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/720 [00:00<?, ?it/s]

Sort by MRR:

In [45]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
7,1.0,4.0,0.2,0.979167,0.875208
3,1.0,2.0,0.1,0.976389,0.873588
19,2.0,4.0,0.2,0.976389,0.873588
35,5.0,10.0,0.5,0.976389,0.873588
6,1.0,4.0,0.1,0.977778,0.872546
4,1.0,2.0,0.2,0.975000,0.872269
8,1.0,4.0,0.5,0.976389,0.872037
20,2.0,4.0,0.5,0.973611,0.870370
18,2.0,4.0,0.1,0.973611,0.868773
34,5.0,10.0,0.2,0.973611,0.868079


The first three rows have the same relative weights. So we can use the smaller and easier-to-read values

In [46]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Usually we care about both metrics. Hit Rate tells us whether the
correct document appears at all. MRR tells us whether it appears near
the top. A document near the top is more likely to be used by the RAG
prompt.